# Lineup Service & Fetcher 測試

測試 `lineup_service` 和 `lineup_fetcher` 的函數，以及 `csv_player_history` 的星級隊友篩選功能。

**執行前**：請在專案根目錄執行，或確保 `backend` 在 Python path 中。

In [1]:
# 設定 Python path，讓 notebook 能 import app 模組
import sys
from pathlib import Path

project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent
backend_path = project_root / "backend"

if str(backend_path) not in sys.path:
    sys.path.insert(0, str(backend_path))

print(f"Project root: {project_root}")
print(f"Backend path: {backend_path}")

Project root: /Users/wuyusen/Documents/bet
Backend path: /Users/wuyusen/Documents/bet/backend


## 1. LineupService 測試

`get_players_in_game(team, date)`：查詢該場該隊出賽球員。
資料來源：優先 lineup JSON，fallback 至 CSV 衍生。

In [3]:
from datetime import datetime
from app.services.lineup_service import lineup_service

# Test Case 1: Bucks 2026-02-24（CSV 有資料）
players = lineup_service.get_players_in_game("Bucks", datetime(2026, 2, 24))
print("Test 1 - Bucks 2026-02-24:")
print(f"  出賽人數: {len(players)}")
print(f"  球員: {sorted(players)}")
assert len(players) >= 1, "Bucks 該日應至少有 1 人出賽"
assert "A.J. Green" in players, "A.J. Green 該日有出賽"
print("  ✓ PASS")

Test 1 - Bucks 2026-02-24:
  出賽人數: 9
  球員: ['A.J. Green', 'Bobby Portis', 'Cam Thomas', 'Jericho Sims', 'Kevin Porter, Jr.', 'Kyle Kuzma', 'Myles Turner', 'Ousmane Dieng', 'Ryan Rollins']
  ✓ PASS


In [4]:
# Test Case 2: Celtics 2026-02-25
players = lineup_service.get_players_in_game("Celtics", datetime(2026, 2, 25))
print("Test 2 - Celtics 2026-02-25:")
print(f"  出賽人數: {len(players)}")
print(f"  球員: {sorted(players)[:5]}..." if len(players) > 5 else f"  球員: {sorted(players)}")
print("  ✓ PASS")

Test 2 - Celtics 2026-02-25:
  出賽人數: 12
  球員: ['Baylor Scheierman', 'Dalano Banton', 'Derrick White', 'Hugo Gonzalez', 'Jaylen Brown']...
  ✓ PASS


In [5]:
# Test Case 3: 不存在的日期/球隊 → 應回傳空 set
players = lineup_service.get_players_in_game("Bucks", datetime(1999, 1, 1))
print("Test 3 - Bucks 1999-01-01 (無資料):")
print(f"  出賽人數: {len(players)}")
assert len(players) == 0, "無資料應回傳空 set"
print("  ✓ PASS")

Test 3 - Bucks 1999-01-01 (無資料):
  出賽人數: 0
  ✓ PASS


In [6]:
# Test Case 4: 驗證 CSV cache 正確性 - 同一場兩隊都應有球員
bucks = lineup_service.get_players_in_game("Bucks", datetime(2026, 2, 24))
heat = lineup_service.get_players_in_game("Heat", datetime(2026, 2, 24))
print("Test 4 - Bucks vs Heat 2026-02-24:")
print(f"  Bucks: {len(bucks)} 人")
print(f"  Heat: {len(heat)} 人")
assert len(bucks) > 0 and len(heat) > 0, "兩隊都應有出賽球員"
print("  ✓ PASS")

Test 4 - Bucks vs Heat 2026-02-24:
  Bucks: 9 人
  Heat: 11 人
  ✓ PASS


## 2. LineupFetcher 測試

測試 `load_lineup_store`、`save_lineup_store`、`_team_to_csv`。
`fetch_day_lineup` 需網路，可選執行。

In [7]:
from app.services import lineup_fetcher

# Test Case 5: _team_to_csv 隊名對應
assert lineup_fetcher._team_to_csv("MILWAUKEE BUCKS") == "Bucks"
assert lineup_fetcher._team_to_csv("BOSTON CELTICS") == "Celtics"
assert lineup_fetcher._team_to_csv("PHILADELPHIA_76ERS") == "Sixers"
assert lineup_fetcher._team_to_csv(None) == ""
print("Test 5 - _team_to_csv: ✓ PASS")

Test 5 - _team_to_csv: ✓ PASS


In [9]:
# Test Case 6: load_lineup_store（無檔案時回傳空 dict）
store = lineup_fetcher.load_lineup_store()
print("Test 6 - load_lineup_store:")
print(f"  類型: {type(store)}")
print(f"  日期數: {len(store)}")
assert isinstance(store, dict), "應回傳 dict"
print("  ✓ PASS")

Test 6 - load_lineup_store:
  類型: <class 'dict'>
  日期數: 26
  ✓ PASS


In [ ]:
# Test Case 7: save_lineup_store + load  roundtrip
import tempfile
import os

original_path = os.environ.get("LINEUP_DATA_PATH")
with tempfile.NamedTemporaryFile(suffix=".json", delete=False) as f:
    test_path = f.name

try:
    os.environ["LINEUP_DATA_PATH"] = test_path
    test_data = {
        "2026-02-24": {"Bucks": ["A.J. Green", "Giannis"], "Heat": ["Jimmy Butler"]}
    }
    lineup_fetcher.save_lineup_store(test_data)
    loaded = lineup_fetcher.load_lineup_store()
    assert loaded == test_data, "save/load roundtrip 應一致"
    print("Test 7 - save/load roundtrip: ✓ PASS")
finally:
    if original_path:
        os.environ["LINEUP_DATA_PATH"] = original_path
    else:
        os.environ.pop("LINEUP_DATA_PATH", None)
    os.unlink(test_path)

## 3. fetch_day_lineup（需網路）

**可選**：會呼叫 Basketball Reference API，需網路且受 rate limit 限制。

In [ ]:
# Test Case 8: fetch_day_lineup（需網路，可跳過）
RUN_NETWORK_TEST = False  # 設為 True 執行

if RUN_NETWORK_TEST:
    day_lineup = lineup_fetcher.fetch_day_lineup(24, 2, 2026)
    print("Test 8 - fetch_day_lineup 2026-02-24:")
    for team, players in day_lineup.items():
        print(f"  {team}: {len(players)} 人")
    assert len(day_lineup) >= 2, "當日應至少有 2 隊比賽"
    print("  ✓ PASS")
else:
    print("Test 8 - fetch_day_lineup: SKIPPED (RUN_NETWORK_TEST=False)")

## 4. CSVPlayerHistoryService + 星級隊友篩選

測試 `get_player_stats` 搭配 `teammate_filter`、`teammate_played`。

In [ ]:
from app.services.csv_player_history import csv_player_service

# Test Case 9: A.J. Green 全部場次
stats_all = csv_player_service.get_player_stats(
    player_name="A.J. Green",
    metric="points",
    threshold=10.5
)
print("Test 9 - A.J. Green 全部場次:")
print(f"  場次: {stats_all['n_games']}")
print(f"  平均: {stats_all['mean']}")
print(f"  Over 機率: {stats_all['p_over']}")
assert stats_all["n_games"] > 0
print("  ✓ PASS")

In [ ]:
# Test Case 10: A.J. Green「Giannis 未上場」的場次
stats_without = csv_player_service.get_player_stats(
    player_name="A.J. Green",
    metric="points",
    threshold=10.5,
    teammate_filter=["Giannis Antetokounmpo"],
    teammate_played=False
)
print("Test 10 - A.J. Green Without Giannis:")
print(f"  場次: {stats_without['n_games']}")
print(f"  平均: {stats_without['mean']}")
print(f"  Over 機率: {stats_without['p_over']}")
assert stats_without["n_games"] <= stats_all["n_games"], "Without 場次應較少或相等"
print("  ✓ PASS")

In [ ]:
# Test Case 11: A.J. Green「Giannis 有上場」的場次
stats_with = csv_player_service.get_player_stats(
    player_name="A.J. Green",
    metric="points",
    threshold=10.5,
    teammate_filter=["Giannis Antetokounmpo"],
    teammate_played=True
)
print("Test 11 - A.J. Green With Giannis:")
print(f"  場次: {stats_with['n_games']}")
print(f"  平均: {stats_with['mean']}")
assert stats_with["n_games"] + stats_without["n_games"] <= stats_all["n_games"] + 5  # 容許少許 overlap 因名稱匹配
print("  ✓ PASS")

In [ ]:
# Test Case 12: 多位星級隊友 - Without Giannis AND Khris
stats_multi = csv_player_service.get_player_stats(
    player_name="A.J. Green",
    metric="points",
    threshold=8.5,
    teammate_filter=["Giannis Antetokounmpo", "Khris Middleton"],
    teammate_played=False
)
print("Test 12 - A.J. Green Without Giannis & Khris:")
print(f"  場次: {stats_multi['n_games']}")
print(f"  平均: {stats_multi['mean']}")
assert stats_multi["n_games"] <= stats_without["n_games"], "雙人篩選場次應更少"
print("  ✓ PASS")

In [ ]:
# Test Case 13: 不存在的球員
stats_unknown = csv_player_service.get_player_stats(
    player_name="NonExistentPlayer123",
    metric="points",
    threshold=10.5
)
print("Test 13 - 不存在的球員:")
print(f"  n_games: {stats_unknown['n_games']}")
print(f"  message: {stats_unknown.get('message')}")
assert stats_unknown["n_games"] == 0
print("  ✓ PASS")

## 5. 總結

| Test | 說明 |
|------|------|
| 1 | LineupService - Bucks 2026-02-24 |
| 2 | LineupService - Celtics 2026-02-25 |
| 3 | LineupService - 無資料回傳空 set |
| 4 | LineupService - 同場兩隊都有球員 |
| 5 | LineupFetcher - _team_to_csv 對應 |
| 6 | LineupFetcher - load_lineup_store |
| 7 | LineupFetcher - save/load roundtrip |
| 8 | LineupFetcher - fetch_day_lineup（需網路） |
| 9 | CSVPlayerHistory - 全部場次 |
| 10 | CSVPlayerHistory - Without 星級隊友 |
| 11 | CSVPlayerHistory - With 星級隊友 |
| 12 | CSVPlayerHistory - 多位星級隊友 |
| 13 | CSVPlayerHistory - 不存在的球員 |